# Regresión Polinómica y Multivariable

## Motivación

La **regresión lineal** asume que la relación entre las variables de entrada y la variable objetivo es una línea recta. Sin embargo, muchos fenómenos del mundo real presentan relaciones **no lineales**.

Por ejemplo, el consumo de un hogar en función de sus ingresos puede crecer de forma curva: al principio crece rápido, luego se estabiliza. Una recta no puede capturar ese comportamiento.

La **regresión polinómica** extiende la regresión lineal permitiendo ajustar curvas de mayor complejidad, manteniendo la misma maquinaria matemática de la regresión lineal (ecuación normal, función de coste, etc.).

## La Matriz de Diseño

### Regresión polinómica (1 variable)

Dado un conjunto de $n$ puntos $(x_i, y_i)$, queremos ajustar un polinomio de orden $r$:

$$h(x) = w_0 + w_1 x + w_2 x^2 + \cdots + w_r x^r = \sum_{j=0}^{r} w_j \, x^j$$

Para escribir esto en forma matricial, construimos la **matriz de diseño** $\mathbf{X}$ de dimensión $(n \times d)$ donde $d = r + 1$:

$$\mathbf{X} = \begin{pmatrix} 1 & x_1 & x_1^2 & \cdots & x_1^r \\ 1 & x_2 & x_2^2 & \cdots & x_2^r \\ \vdots & \vdots & \vdots & \ddots & \vdots \\ 1 & x_n & x_n^2 & \cdots & x_n^r \end{pmatrix}$$

Y el vector de pesos $\mathbf{w} = (w_0, w_1, \ldots, w_r)$. La hipótesis completa para todos los puntos es simplemente:

$$\mathbf{h} = \mathbf{X} \, \mathbf{w}^\top$$

### Regresión multivariable

Cuando el problema tiene **múltiples variables de entrada** $(x^{(1)}, x^{(2)}, \ldots, x^{(p)})$, la matriz de diseño incorpora una columna por cada variable (y sus potencias si también se quiere modelar no linealidades en cada una):

$$\mathbf{X} = \begin{pmatrix} 1 & x_1^{(1)} & x_1^{(2)} & \cdots & x_1^{(p)} \\ \vdots & \vdots & \vdots & \ddots & \vdots \\ 1 & x_n^{(1)} & x_n^{(2)} & \cdots & x_n^{(p)} \end{pmatrix}$$

La estructura matemática es idéntica: sigue siendo un problema lineal **en los pesos** $\mathbf{w}$, aunque sea no lineal en las variables originales. Esto es lo que hace que la regresión polinómica sea un caso especial de regresión lineal.

## Optimización: Ecuación Normal con Regularización de Tikhonov

### Función de coste (sin regularización)

Queremos encontrar los pesos $\mathbf{w}$ que minimicen el error cuadrático medio entre la hipótesis y los valores reales:

$$J(\mathbf{w}) = \frac{1}{n} \|\mathbf{X}\mathbf{w}^\top - \mathbf{y}\|^2 = \frac{1}{n} \sum_{i=1}^{n}(h(x_i) - y_i)^2$$

### Ecuación normal

Derivando $J$ respecto a $\mathbf{w}$ e igualando a cero, se obtiene la **ecuación normal**:

$$\mathbf{w}^\top = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

Esta solución es **exacta y analítica**: no requiere iteraciones. La matriz $\mathbf{C} = \mathbf{X}^\top \mathbf{X}$ se llama **matriz de covarianza** (o de Gram).

### Regularización de Tikhonov (Ridge)

Con modelos de alto orden, la matriz $\mathbf{C}$ puede volverse **casi singular** (mal condicionada), y los pesos tienden a dispararse, produciendo **sobreajuste** (*overfitting*).

La regularización de Tikhonov añade un término de penalización que controla la magnitud de los pesos:

$$J_\lambda(\mathbf{w}) = \frac{1}{n} \|\mathbf{X}\mathbf{w}^\top - \mathbf{y}\|^2 + \lambda \|\mathbf{w}\|^2$$

La nueva ecuación normal regularizada queda:

$$\mathbf{w}^\top = (\mathbf{X}^\top \mathbf{X} + n\lambda \mathbf{I})^{-1} \mathbf{X}^\top \mathbf{y}$$

donde $\lambda \geq 0$ es el **coeficiente de regularización**:
- $\lambda = 0$: sin regularización (puede sobreajustar).
- $\lambda$ grande: penaliza pesos grandes, el modelo se vuelve más simple (puede subajustar).

## Función de Coste: MSE

La función `CalculaCoste` evalúa el **Error Cuadrático Medio** (*Mean Squared Error*, MSE):

$$J(\mathbf{w}) = \frac{1}{n} \sum_{i=1}^{n} \left( h(x_i) - y_i \right)^2$$

donde $h(x_i) = \mathbf{x}_i \cdot \mathbf{w}^\top$ es la predicción del modelo para el punto $i$.

Se calcula tanto sobre el **conjunto de entrenamiento** (*training set*) como sobre el **conjunto de prueba** (*test set*), lo que permite detectar sobreajuste o subajuste.

## Sesgo y Varianza: Selección del Orden del Polinomio

Una de las decisiones más importantes en regresión polinómica es elegir el **orden $r$** del modelo. Hay un compromiso fundamental conocido como el **dilema sesgo-varianza**:

| Situación | Nombre | Síntoma |
|-----------|--------|---------|
| Orden muy bajo | **Subajuste** (*underfitting*) | $J_{train}$ alto y $J_{test}$ alto — el modelo es demasiado simple para capturar la estructura de los datos |
| Orden muy alto | **Sobreajuste** (*overfitting*) | $J_{train}$ bajo pero $J_{test}$ alto — el modelo memoriza el ruido del entrenamiento y no generaliza |
| Orden adecuado | **Buen ajuste** | Ambos costes bajos y similares |

La gráfica de **coste vs. orden del polinomio** permite detectar visualmente en qué zona estamos:

- La curva de entrenamiento decrece monótonamente al aumentar $r$ (mayor complejidad → mejor ajuste al training).
- La curva de prueba tiene un **mínimo**: antes de ese mínimo hay subajuste, después hay sobreajuste.

El orden óptimo $r^*$ se encuentra en el mínimo de la curva de prueba.

## Selección del Coeficiente de Regularización $\lambda$

Una vez fijado el orden $r$, hay que elegir el valor de $\lambda$ que mejor equilibre complejidad y generalización. Para esto se usa un **conjunto de validación** (distinto del de entrenamiento y del de prueba).

El procedimiento es:

1. Entrenar el modelo para una grilla de valores de $\lambda$ (típicamente en escala logarítmica: $10^{-4}, \ldots, 10^{4}$).
2. Evaluar $J_{val}$ para cada $\lambda$.
3. Elegir el $\lambda^*$ que minimiza $J_{val}$.

**Efecto de $\lambda$ en el modelo:**

- $\lambda \to 0$: los pesos pueden crecer libremente → riesgo de sobreajuste.
- $\lambda \to \infty$: todos los pesos se acercan a cero → el modelo se vuelve constante (subajuste severo).
- El $\lambda$ óptimo está en algún punto intermedio, y varía según el orden del polinomio y la escala de los datos.

La gráfica **coste vs. $\lambda$** (en escala log-log) muestra este comportamiento para distintos órdenes del polinomio, permitiendo elegir simultáneamente $r$ y $\lambda$.

## Ejecución del Modelo

Se ajusta un polinomio de orden $r = 7$ sobre los primeros 10 puntos del dataset (`ntrain = 10`) y se evalúa en los restantes como conjunto de prueba/validación.

Las dos gráficas generadas muestran:
1. **Coste vs. orden del polinomio** — para encontrar el $r$ óptimo.
2. **Coste vs. $\lambda$** — para encontrar el coeficiente de regularización óptimo para cada orden.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def CargaDatos():
    df = pd.read_excel('DatosRegresion1D.xlsx')
    x = df['ingresos']
    y = df['consumo']
    return x,y

In [ ]:
def GeneraMatrices(x,y,r):
    #x: vector de diseño (ingresos)
    #y: vector de objetivos (target) (consumo)
    #r: Orden del polinomio
    #X: Matriz de diseño
    n = len(x)  #Determina la longitud de x --> número de registros
    d = r+1
    X = np.zeros((n,d))
    for j in range(d):
        X[:,j] = x**j
    y = y.values.reshape(n,1) #Transforma y en un vector de dimensión (n,1)
    return X,y

In [ ]:
def OptimizaModelo(X,y,n,L):
    #X: Matriz de diseño
    #y: vector de objetivos (target)
    #n: nº de puntos a considerar para la optimización
    #L: coeficiente de regularización (Tikhonov)
    #w: coeficientes del modelo

    d = X.shape[1]
    X = X[:n,:]
    y = y[:n]
    C = X.T @ X #Matriz de covarianza
    Id = np.identity(d) #Matriz de identidad (dxd)
    Cr = C + n*L*Id #Matriz de covarianza regularizada
    w = y.T @ X @ np.linalg.inv(Cr) #Ecuación normal para cálcuclo del óptimo
    w = w.reshape(1,d) #Transforma w en un vector de dimensión (1,d)
    return w

In [ ]:
def CalculaCoste(X,y,w):
    #X: Matriz de diseño
    #y: vector de objetivos (target)
    #w: coeficientes del modelo
    #J: Valor del coste

    n = len(y) #Nº de elementos
    h = X @ w.T #hipótesis
    e = h-y #error (h-y)
    J = np.sum(e**2) / n #Coste
    return J

In [ ]:
def DibujaCosteVsComplejidad(X,y,ntrain,L):
    #X: Matriz de diseño
    #y: vector de objetivos (target)
    #ntrain: nº de puntos del training dataset
    #L: coeficiente de regularización (Tikhonov)

    r = X.shape[1]-1
    r_vector = np.linspace(1,r,r) #Orden del polinomio (vector)
    npuntos = len(r_vector) #Nº de puntos de la gráfica
    Jtrain = np.zeros(npuntos) #Reserva de espacio para el coste de training
    Jtest = np.zeros(npuntos) #Reserva de espacio para el coste de testing

    for k in range(npuntos): #Itera para distintos valores de r
        #Cálculo del coste para el Training Dataset
        r = r_vector[k]
        r = r.astype('int') #Lo convierte en un número entero
        Xtrain = X[:ntrain,:r+1] #Vector de diseño (training)
        ytrain = y[:ntrain] #Vector de objetivos (training)
        w = OptimizaModelo(Xtrain,ytrain,ntrain,L) #w óptima
        Jtrain[k] = CalculaCoste(Xtrain,ytrain,w)

        #Cálculo del coste para el Test Dataset
        Xtest = X[ntrain:,:r+1] #Vector de diseño (test)
        ytest = y[ntrain:] #Vector de objetivos (test)
        Jtest[k] = CalculaCoste(Xtest,ytest,w)

    #Dibujo de la curva de aprendizaje
    plt.figure()
    plt.semilogy(r_vector,Jtrain, label='Entrenamiento')
    plt.semilogy(r_vector,Jtest, label='Prueba')
    plt.xlabel('Orden del polinomio')
    plt.ylabel('Función de coste: J(w)')
    plt.legend()

In [ ]:
def DibujaCosteVsLambda(X,y,ntrain,nval):
    #X: Matriz de diseño
    #y: vector de objetivos (target)
    #ntrain: nº de puntos del training dataset
    #L: coeficiente de regularización (Tikhonov)

    r = X.shape[1]-1
    r_vector = np.linspace(1,r,r) #Orden del polinomio (vector)
    ngraf = len(r_vector) #Nº de gráficas
    L_vector = np.logspace(-4,4,100) #Coeficiente de regularización (vector)
    npuntos = len(L_vector) #Nº de puntos de la gráfica
    Jval = np.zeros(npuntos) #Reserva de espacio para el coste de validación

    plt.figure()
    for ir in range(ngraf): #Itera para distintos valores de r
        r = r_vector[ir] #orden del polinomio
        r = r.astype('int') #Lo convierte en un número entero
        for k in range(npuntos): #Itera para distintos valores de L
            #Cálculo del coste para el Training Dataset
            L = L_vector[k] #Coeficiente de regularización
            Xtrain = X[:ntrain,:r+1] #Vector de diseño (training)
            ytrain = y[:ntrain] #Vector de objetivos (training)
            w = OptimizaModelo(Xtrain,ytrain,ntrain,L) #w óptima

            #Cálculo del coste para el Validation Dataset
            Xval = X[ntrain:ntrain+nval,:r+1] #Vector de diseño (validación)
            yval = y[ntrain:ntrain+nval] #Vector de objetivos (validación)
            Jval[k] = CalculaCoste(Xval,yval,w)

        #Dibujo de la curva de aprendizaje
        plt.loglog(L_vector,Jval, label=str(r))
    plt.xlabel('Coeficiente de regularización')
    plt.ylabel('Función de coste: J(w)')
    plt.legend()

In [ ]:
x,y = CargaDatos()
X,y = GeneraMatrices(x,y,7)
w = OptimizaModelo(X,y,10,0)
print(w)
DibujaCosteVsComplejidad(X,y,10,0.1)
DibujaCosteVsLambda(X,y,10,500)
plt.show()